In [ ]:
!pip -q install --upgrade pip
!pip -q install openai datasets pillow jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.1 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import drive
from google.colab import userdata
from google import genai
from google.genai import types
from datasets import load_dataset
from PIL import Image
from IPython.display import display
import pandas as pd
from jiwer import wer, cer
from tqdm.auto import tqdm
import json
import time

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from google.colab import userdata
from google import genai
import importlib.metadata # Import to check package version

os.environ['GOOGLE_API_KEY'] = userdata.get('GEMINI_API_KEY')

# Print the installed version of google-generativeai for verification
try:
    print(f"google-generativeai version: {importlib.metadata.version('google-generativeai')}")
except importlib.metadata.PackageNotFoundError:
    print("google-generativeai package not found. Please install it.")

# Initialize the Gemini client globally
client = genai.Client()

google-generativeai version: 0.8.5


In [ ]:
BASE_DIR    = "/content/drive/MyDrive/Arabic-OCR-Project"

# Define the dataset name for organizing images and results
DATASET_NAME = "muharaf"

IMAGES_DIR  = os.path.join(BASE_DIR, "images", DATASET_NAME)
RESULTS_DIR = os.path.join(BASE_DIR, "results", DATASET_NAME)

# Change this variable to create new subfolders for different runs
RUN_NAME = "full_data_run"
RUN_RESULTS_DIR = os.path.join(RESULTS_DIR, RUN_NAME)

# Create necessary directories
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(RUN_RESULTS_DIR, exist_ok=True)

PROMPT = (
    "You are an OCR system. Read the Arabic text in the image and output it exactly as it appears. "
    "Do not translate, explain, or add anything. Output only the text."
)

CONFIG = {
    "model": "gemini-2.5-flash",
    "temperature": 0.0,
    "prompt": PROMPT,
}

In [ ]:
def test_wer_cer(ref: str, hyp: str):
    wer_val = wer(ref, hyp)
    cer_val = cer(ref, hyp)
    return wer_val, cer_val

In [ ]:
def verify_usage_fields(model_name: str):
  resp = client.models.generate_content(
      model=model_name,
      contents="test"
  )
  usage = getattr(resp, "usage_metadata", None)
  print("usage_metadata:", usage)
  if usage:
    print("Type:", type(usage))
    # Ensure usage is a dictionary or has keys/attrs attribute before accessing
    if isinstance(usage, dict):
        print("Available keys:", usage.keys())
    else:
        print("Available attributes:", dir(usage))

verify_usage_fields(model_name=CONFIG["model"])

usage_metadata: cache_tokens_details=None cached_content_token_count=None candidates_token_count=18 candidates_tokens_details=None prompt_token_count=2 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=2
)] thoughts_token_count=550 tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=570 traffic_type=None
Type: <class 'google.genai.types.GenerateContentResponseUsageMetadata'>
Available attributes: ['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '_

Candidates token means the number of token for the prediction text

In [ ]:
def extract_usage(resp):
    """Extract usage info from a Gemini response (protobuf-safe)."""
    usage = getattr(resp, "usage_metadata", None)
    if not usage:
        return 0, 0, 0  # prompt, output, total

    # Works for both dict-like and protobuf-like usage objects
    prompt = getattr(usage, "prompt_token_count", 0)
    output = getattr(usage, "candidates_token_count", 0)
    total = getattr(usage, "total_token_count", 0)

    return prompt, output, total


In [ ]:
def ocr_gemini(image_path, prompt: str, model_name: str, temperature=0.0):
    img = Image.open(image_path).convert("RGB")

    start = time.perf_counter()

    resp = None
    max_retries = 3
    fixed_delay_seconds = 5

    for attempt in range(max_retries):
        try:
            resp = client.models.generate_content(
                model=model_name,
                contents=[img, prompt],
                config=types.GenerateContentConfig(
                    temperature=temperature
                ),)

            if resp and resp.candidates and resp.candidates[0].content.parts:
                break
            else:
                print(f"Attempt {attempt + 1} for {image_path}: Retrying...")
                raise ValueError("Empty response or no content parts")

        except Exception as e:
            print(f"Attempt {attempt + 1} for {image_path} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(fixed_delay_seconds)
            else:
                print(f"All {max_retries} attempts failed for {image_path}.")
                # Return default values if all retries fail
                return "", 0, 0, 0, 0

    end = time.perf_counter()
    latency = end - start

    prompt_tokens, output_tokens, total_tokens = extract_usage(resp)

    # pred will be an empty string if resp is None or has no content parts after retries
    pred = (resp.text or "").strip() if resp and resp.candidates and resp.candidates[0].content.parts else ""

    return pred , latency, prompt_tokens, output_tokens, total_tokens

There is no seprate charge for images in Gemini they treated like the (prompt) so the images cost included in input token price

---

The visual information in the image converted into numerical representation that the model can understand.

---

This numerical representation is then treated as part of the input sequesnce just like the the tokens derived from the text prompt

In [ ]:
def compute_cost(prompt_tokens, output_tokens, images, model_name):
    PRICING = {
        "gemini-2.5-flash": {
            "input_token_price": 0.30 / 1000000,
            "output_token_price": 2.50 / 1000000,
            "image_price": 0.0
        },
       }

    if model_name not in PRICING:
        raise ValueError(f"Model {model_name} not found in pricing dict")

    cost = (prompt_tokens * PRICING[model_name]["input_token_price"]) + \
           (output_tokens * PRICING[model_name]["output_token_price"]) + \
           (images * PRICING[model_name]["image_price"])

    return cost

In [ ]:
def evaluate(ref: str, hyp: str) -> dict:
  wer_val, cer_val = test_wer_cer(ref, hyp)
  return {
      "CER": cer_val,
      "WER": wer_val,
      "Accuracy": 1 - cer_val
  }

Global cer and wer computing by concatenating all the ground truth texts and all the predicted texts from the entire batch.

In [ ]:
def run_batch(items, out_csv_path: str, show_images=True):
    rows = []
    processed_items = set()

    # Check for existing results to resume
    if os.path.exists(out_csv_path):
        existing_df = pd.read_csv(out_csv_path)
        rows = existing_df.to_dict('records')
        processed_items = set(existing_df["image"].tolist())
        print(f"Resuming batch. Found {len(processed_items)} already processed images.")

    # Process items
    for item in tqdm(items, desc="Processing images"):
        image_filename = os.path.basename(item["path"])
        if image_filename in processed_items:
            continue

        if show_images:
            display(Image.open(item["path"]).convert("RGB"))

        pred, latency, p_tok, o_tok, t_tok = ocr_gemini(
            item["path"],
            prompt=CONFIG["prompt"],
            model_name=CONFIG["model"],
            temperature=CONFIG["temperature"],
        )

        pred = str(pred) if pred is not None else ""

        metrics = evaluate(item["gt"], pred)

        item_cost = compute_cost(
            p_tok, o_tok, 1,
            model_name=CONFIG['model']
        )

        rows.append({
            "image": image_filename,
            "ground_truth": item["gt"],
            "prediction": pred,
            "CER": metrics["CER"],
            "WER": metrics["WER"],
            "Accuracy": metrics["Accuracy"],
            "Latency_s": latency,
            "Prompt_Tokens": p_tok,
            "Output_Tokens": o_tok,
            "Total_Tokens": t_tok,
            "Images": 1,
            "Cost_USD": item_cost,
            "model": CONFIG['model'],
        })

        # Save incrementally to allow resume
        pd.DataFrame(rows).to_csv(out_csv_path, index=False, encoding="utf-8")

    df = pd.DataFrame(rows)
    print("Saved:", out_csv_path)

    return df


In [ ]:
subset_dir = os.path.join(IMAGES_DIR, f"{DATASET_NAME}_test")
os.makedirs(subset_dir, exist_ok=True)

ds = load_dataset("aamijar/muharaf-public", split="test")

muh_items = []
for i, item in enumerate(ds):
    image = item["image"]
    image_path = os.path.join(subset_dir, f"image_{i}.png")
    if not os.path.exists(image_path):
        image.save(image_path)
    muh_items.append({
        "path": image_path,
        "gt": item["text"]
    })

CHUNK_SIZE = 100
all_results = []

# Construct model-specific filename part
model_filename_part = CONFIG['model'].replace('.', '_').replace('-', '_')

# Use the run-specific results directory
final_csv = os.path.join(RUN_RESULTS_DIR, f"test_{model_filename_part}_full.csv")
if os.path.exists(final_csv):
    existing_df = pd.read_csv(final_csv)
    all_results.append(existing_df)
    print(f"Loaded existing results from {final_csv}")
    processed_image_filenames = set(existing_df["image"].tolist())
else:
    processed_image_filenames = set()


remaining_items = [
    item for item in muh_items
    if os.path.basename(item["path"]) not in processed_image_filenames
]

print(f"Starting processing for {len(remaining_items)} remaining images.")

start_batch_idx = len(muh_items) - len(remaining_items)

for start_idx in range(start_batch_idx, len(muh_items), CHUNK_SIZE):
    end_idx = start_idx + CHUNK_SIZE
    batch = muh_items[start_idx:end_idx]

    batch_to_process = [
        item for item in batch
        if os.path.basename(item["path"]) not in processed_image_filenames
    ]

    if not batch_to_process:
        print(f"Batch {start_idx}–{end_idx} already processed. Skipping.")
        continue

    print(f"Processing batch {start_idx}–{end_idx} of {len(muh_items)}")

    # Use the run-specific results directory for batch CSVs
    batch_csv = os.path.join(
        RUN_RESULTS_DIR, f"test_{model_filename_part}_batch_{start_idx}.csv"
    )

    batch_results = run_batch(batch_to_process, batch_csv, show_images=False)

    all_results.append(batch_results)

# Concatenate all results after the loop
if all_results:
    results_df = pd.concat(all_results, ignore_index=True)
    results_df.to_csv(final_csv, index=False)
    display(results_df.head())
    print(f"Benchmark complete — results saved to {final_csv}")
else:
    print("No new images processed.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/385M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/399M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/403M [00:00<?, ?B/s]

...

Processing images:   0%|          | 0/100 [00:00<?, ?it/s]

Attempt 1 for /content/drive/MyDrive/Arabic-OCR-Project/images/muharaf/muharaf_test/image_13.png failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 1 for /content/drive/MyDrive/Arabic-OCR-Project/images/muharaf/muharaf_test/image_25.png failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 1 for /content/drive/MyDrive/Arabic-OCR-Project/images/muharaf/muharaf_test/image_41.png failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 1 for /content/drive/MyDrive/Arabic-OCR-Project/images/muharaf/muharaf_test/image_42.png failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 1 for /content/drive/MyDrive/Arabic-OCR-Project/imag

Processing images:   0%|          | 0/100 [00:00<?, ?it/s]

Attempt 1 for /content/drive/MyDrive/Arabic-OCR-Project/images/muharaf/muharaf_test/image_102.png failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 1 for /content/drive/MyDrive/Arabic-OCR-Project/images/muharaf/muharaf_test/image_103.png failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 1 for /content/drive/MyDrive/Arabic-OCR-Project/images/muharaf/muharaf_test/image_110.png failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 1 for /content/drive/MyDrive/Arabic-OCR-Project/images/muharaf/muharaf_test/image_111.png failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}
Attempt 1 for /content/drive/MyDrive/Arabic-OCR-Project/

KeyboardInterrupt: 

In [ ]:
# Concatenate all results after the loop
if all_results:
    results_df = pd.concat(all_results, ignore_index=True)
    results_df.to_csv(final_csv, index=False)
    display(results_df.head())
    print(f"Benchmark complete — results saved to {final_csv}")
else:
    print("No new images processed.")


# === Global summary over the FULL dataset ===

# Ensure results_df exists (e.g. if you only want to compute summary later)
if 'results_df' not in globals():
    # Construct model-specific filename part
    model_filename_part = CONFIG['model'].replace('.', '_').replace('-', '_')
    # Updated to use RUN_RESULTS_DIR and dynamic filename
    final_csv = os.path.join(RUN_RESULTS_DIR, f"test_{model_filename_part}_full.csv")
    results_df = pd.read_csv(final_csv)

print(f"Total rows in results: {len(results_df)}")

gt_series = results_df["ground_truth"].astype(str)
pred_series = results_df["prediction"].fillna("").astype(str)

global_ref = " ".join(gt_series.tolist())
global_hyp = " ".join(pred_series.tolist())

global_cer = cer(global_ref, global_hyp)
global_wer = wer(global_ref, global_hyp)
global_acc = 1 - global_cer

# totals
total_prompt = results_df["Prompt_Tokens"].sum()
total_output = results_df["Output_Tokens"].sum()
total_tokens = results_df["Total_Tokens"].sum()
total_images = results_df["Images"].sum()

total_cost = compute_cost(
    total_prompt,
    total_output,
    total_images,
    model_name=CONFIG["model"]
)

summary = {
    "Global_CER": global_cer,
    "Global_WER": global_wer,
    "Global_Acc": global_acc,
    "Total_Prompt_Tokens": total_prompt,
    "Total_Output_Tokens": total_output,
    "Total_Tokens": total_tokens,
    "Total_Images": total_images,
    "Estimated_Total_Cost_USD": total_cost,
    "model": CONFIG["model"],
}

summary_df = pd.DataFrame([summary])

# Save as a global summary CSV next to the full results
global_summary_csv = final_csv.replace(".csv", "_global_summary.csv")
summary_df.to_csv(global_summary_csv, index=False, encoding="utf-8")

print("\n=== Global summary over all Muharaf test images ===")
display(summary_df)
print(f"Global summary saved to: {global_summary_csv}")